# VisionAid controlled Pillar study
Attach the existing private VisionAid Pillar Pilot v4 Input. Enable GPU and Internet.
Three matched 10-epoch experiments. All saved epochs are evaluated on validation only.
No candidate is deployed automatically; preserve the review ZIP and complete external checks.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil, hashlib, time
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.183"],check=True)
import torch
assert torch.cuda.is_available(), "GPU required; CPU fallback disabled"
print("STUDY_GPU",torch.cuda.get_device_name(0),flush=True)
Path('pillar_gpu_pilot.py').write_text('"""Ten-epoch GPU pilot; training-only scene-capped pillar oversampling.\n\nRun on Kaggle with the prepared private input bundle. No test images are included\nor evaluated. A candidate is never promoted automatically to the application.\n"""\nimport csv\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nNAMES = [\'Escalator\', \'People\', \'Pillar\', \'Stair\', \'Wet Floor Sign\']\nEPOCHS = 10\nSEED = 0\n\n\ndef iou(a, b):\n    overlap = max(0, min(a[2], b[2])-max(a[0], b[0])) * max(0, min(a[3], b[3])-max(a[1], b[1]))\n    area = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - overlap\n    return overlap / area if area else 0\n\n\ndef evaluate(model, root, output, tag):\n    """Deployment-threshold greedy matches, plus low-threshold AP."""\n    import yaml\n    config = output / \'baseline.yaml\'\n    config.write_text(yaml.safe_dump(dict(path=str(root), train=\'train/images\', val=\'valid/images\',\n                                         nc=5, names=NAMES)))\n    metrics = model.val(data=str(config), split=\'val\', imgsz=640, batch=16, conf=.001,\n                        iou=.7, device=0, plots=False, project=str(output), name=tag+\'_ap\')\n    tp, fp, fn = Counter(), Counter(), Counter()\n    images = sorted((root / \'valid/images\').glob(\'*\'))\n    # A Python image list uses LoadPilAndNumpy, which batches the entire list.\n    # Directory input uses the file loader and respects the bounded batch size.\n    for prediction in model.predict(source=str(root / \'valid/images\'), batch=16, stream=True, imgsz=640,\n                                    conf=.35, iou=.7, rect=False, device=0, verbose=False):\n        image = Path(prediction.path)\n        height, width = prediction.orig_shape\n        truths = defaultdict(list)\n        label = root / \'valid/labels\' / (image.stem+\'.txt\')\n        for line in label.read_text().splitlines():\n            cid, x, y, w, h = map(float, line.split())\n            truths[int(cid)].append(((x-w/2)*width, (y-h/2)*height, (x+w/2)*width, (y+h/2)*height))\n        used = defaultdict(set)\n        for x1,y1,x2,y2,confidence,cid in sorted(prediction.boxes.data.cpu().tolist(),key=lambda r:-r[4]):\n            cid = int(cid)\n            available = [j for j in range(len(truths[cid])) if j not in used[cid]]\n            best = max(available,key=lambda j:iou((x1,y1,x2,y2),truths[cid][j]),default=None)\n            if best is not None and iou((x1,y1,x2,y2),truths[cid][best]) >= .5:\n                used[cid].add(best); tp[cid] += 1\n            else:\n                fp[cid] += 1\n        for cid in range(5):\n            fn[cid] += len(truths[cid])-len(used[cid])\n    report = dict(images=len(images), map50=float(metrics.box.map50), map50_95=float(metrics.box.map),\n                  confidence=.35, per_class={name:dict(tp=tp[c],fp=fp[c],fn=fn[c],\n                  precision=tp[c]/max(tp[c]+fp[c],1), recall=tp[c]/max(tp[c]+fn[c],1)) for c,name in enumerate(NAMES)})\n    (output/(tag+\'.json\')).write_text(json.dumps(report,indent=2))\n    return report\n\n\ndef main():\n    import torch\n    import yaml\n    from ultralytics import YOLO\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU required. Enable a Kaggle GPU; no silent CPU fallback.\')\n    inputs = Path(\'/kaggle/input\')\n    candidates = list(inputs.rglob(\'pilot_input.json\'))\n    if len(candidates) != 1:\n        raise RuntimeError(\'Attach the single prepared VisionAid pillar pilot input bundle\')\n    root = candidates[0].parent\n    metadata = json.loads(candidates[0].read_text())\n    manifest_text = (root/\'split_manifest.csv\').read_text()\n    if hashlib.md5(manifest_text.encode()).hexdigest() != \'49601080aac1f838190341d331785578\':\n        raise RuntimeError(\'Wrong scene split\')\n    rows = list(csv.DictReader(manifest_text.splitlines()))\n    scene_splits = defaultdict(set)\n    for r in rows: scene_splits[r[\'scene\']].add(r[\'split\'])\n    if any(len(s)>1 for s in scene_splits.values()): raise RuntimeError(\'Scene leakage\')\n    if (root/\'test\').exists(): raise RuntimeError(\'Pilot bundle must exclude test images\')\n    train = sorted((root/\'train/images\').glob(\'*\'))\n    valid = sorted((root/\'valid/images\').glob(\'*\'))\n    if len(train)!=8247 or len(valid)!=925: raise RuntimeError(\'Unexpected data counts\')\n    weights = root/\'yolov8n_v4.pt\'\n    if hashlib.sha256(weights.read_bytes()).hexdigest() != metadata[\'weights_sha256\']:\n        raise RuntimeError(\'Checkpoint integrity failed\')\n    output = Path(\'/kaggle/working\') / (\'pillar_pilot_\' + time.strftime(\'%Y%m%d_%H%M%S\'))\n    output.mkdir(exist_ok=False)\n    model = YOLO(str(weights))\n    if list(model.names.values()) != NAMES: raise RuntimeError(\'Wrong model taxonomy\')\n    before = evaluate(model,root,output,\'before_validation\')\n    scenes = {r[\'stem\']:r[\'scene\'] for r in rows if r[\'split\']==\'train\'}\n    pillar_groups = defaultdict(list)\n    for p in train:\n        label = root/\'train/labels\'/(p.stem+\'.txt\')\n        if any(line.split()[0]==\'2\' for line in label.read_text().splitlines() if line.strip()):\n            pillar_groups[scenes[p.stem]].append(p)\n    rng = random.Random(SEED)\n    extra = []\n    for scene in sorted(pillar_groups):\n        members = sorted(pillar_groups[scene]); rng.shuffle(members)\n        extra.extend(members[:50]*2)\n    if not extra: raise RuntimeError(\'No pillar training examples\')\n    train_list = output/\'train_scene_balanced.txt\'\n    train_list.write_text(\'\'.join(str(p)+\'\\n\' for p in train+extra))\n    dataset_yaml = output/\'pilot.yaml\'\n    dataset_yaml.write_text(yaml.safe_dump(dict(path=str(root),train=str(train_list),\n                            val=\'valid/images\',nc=5,names=NAMES)))\n    plan = dict(epochs=EPOCHS,seed=SEED,base_training_images=len(train),extra_pillar_draws=len(extra),\n                extra_cap_per_scene=100,pillar_training_scenes=len(pillar_groups),\n                labels_changed=False,test_used=False,method=\'scene-capped extra pillar exposure\',\n                limitations=\'Pilot does not isolate extra training epochs from resampling; a matched control is needed for causal claims.\')\n    (output/\'plan.json\').write_text(json.dumps(plan,indent=2))\n    print(\'PILOT PLAN\',json.dumps(plan),flush=True)\n    epoch_times=[]\n    def started(trainer): trainer._pilot_epoch_start=time.monotonic()\n    def ended(trainer):\n        epoch_times.append(time.monotonic()-trainer._pilot_epoch_start)\n        remaining=(EPOCHS-trainer.epoch-1)*sum(epoch_times)/len(epoch_times)\n        print(f\'PILOT TIMING epoch={trainer.epoch+1} seconds={epoch_times[-1]:.1f} estimated_remaining_minutes={remaining/60:.1f}\',flush=True)\n    model.add_callback(\'on_train_epoch_start\',started)\n    model.add_callback(\'on_train_epoch_end\',ended)\n    model.train(data=str(dataset_yaml),epochs=EPOCHS,patience=3,batch=16,imgsz=640,\n                device=0,seed=SEED,workers=2,optimizer=\'AdamW\',lr0=.0005,\n                project=str(output),name=\'candidate\',save_period=1,exist_ok=False)\n    candidate = YOLO(str(output/\'candidate/weights/best.pt\'))\n    after = evaluate(candidate,root,output,\'after_validation\')\n    pillar_before,pillar_after = before[\'per_class\'][\'Pillar\'],after[\'per_class\'][\'Pillar\']\n    other_regressions = [name for name in NAMES if name!=\'Pillar\' and\n                         after[\'per_class\'][name][\'recall\'] < before[\'per_class\'][name][\'recall\']-.03]\n    passed = (pillar_after[\'recall\'] >= pillar_before[\'recall\']+.10 and\n              pillar_after[\'precision\'] >= max(.60,pillar_before[\'precision\']-.02) and\n              after[\'map50\'] >= before[\'map50\']-.01 and not other_regressions)\n    result=dict(candidate_passes_pilot_gate=passed,other_class_regressions=other_regressions,\n                before=before,after=after,application_changed=False,epoch_seconds=epoch_times,\n                next_step=\'Review validation examples and run matched control/new field test; never auto-replace deployed weights.\')\n    (output/\'pilot_result.json\').write_text(json.dumps(result,indent=2))\n    print(json.dumps(result,indent=2),flush=True)\n\n\nif __name__==\'__main__\': main()\n')
Path('controlled_pillar_study.py').write_text('"""Matched GPU experiments: control, curated data, and curated scene sampling.\n\nAll arms start at the original checkpoint, use the same number of training\ndraws/updates, and evaluate the unchanged validation split. Test is excluded.\n"""\nimport argparse\nfrom collections import Counter, defaultdict, deque\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nfrom pillar_gpu_pilot import NAMES, evaluate\n\nFINGERPRINT = \'49601080aac1f838190341d331785578\'\nQUARANTINE = {\'IMG_8603_MOV\', \'amel-1_mp4\'}\n\n\ndef acceptance(before, after):\n    b, a = before[\'per_class\'][\'Pillar\'], after[\'per_class\'][\'Pillar\']\n    regressions = [name for name in NAMES if name != \'Pillar\' and\n        after[\'per_class\'][name][\'recall\'] < before[\'per_class\'][name][\'recall\'] - .03]\n    passed = (a[\'recall\'] >= b[\'recall\'] + .10 and\n              a[\'precision\'] >= max(.75, b[\'precision\'] - .02) and\n              after[\'map50\'] >= before[\'map50\'] - .01 and not regressions)\n    return passed, regressions\n\n\ndef balanced_draws(groups, negatives, total, seed, fraction=.25, repeats=4):\n    """Round-robin scene exposure, with a hard cap on repeats per pillar image."""\n    rng = random.Random(seed)\n    target = round(total * fraction)\n    queues = []\n    for scene in sorted(groups):\n        values = list(groups[scene]) * repeats\n        rng.shuffle(values)\n        queues.append(deque(values))\n    chosen = []\n    while len(chosen) < target:\n        progressed = False\n        for queue in queues:\n            if queue and len(chosen) < target:\n                chosen.append(queue.popleft())\n                progressed = True\n        if not progressed:\n            raise ValueError(\'Too few reviewed pillar examples for the requested repeat cap\')\n    negatives = list(negatives)\n    rng.shuffle(negatives)\n    if len(negatives) < total - target:\n        raise ValueError(\'Too few non-pillar examples for the fixed training budget\')\n    chosen += negatives[:total - target]\n    rng.shuffle(chosen)\n    return chosen\n\n\ndef main():\n    import torch\n    import yaml\n    from ultralytics import YOLO\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--arm\', choices=[\'control\', \'curated\', \'curated-balanced\'], required=True)\n    parser.add_argument(\'--seed\', type=int, default=0)\n    parser.add_argument(\'--epochs\', type=int, default=10)\n    parser.add_argument(\'--input\', type=Path, help=\'Prepared private pilot input root; otherwise discover on Kaggle\')\n    parser.add_argument(\'--output\', type=Path, default=Path(\'/kaggle/working\'))\n    args = parser.parse_args()\n    if args.epochs < 1:\n        parser.error(\'--epochs must be positive\')\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU required; no silent CPU fallback\')\n    roots = [args.input] if args.input else [p.parent for p in Path(\'/kaggle/input\').rglob(\'pilot_input.json\')]\n    if len(roots) != 1:\n        raise ValueError(\'Attach exactly the prepared v4 private pilot input\')\n    root = roots[0].resolve()\n    metadata = json.loads((root / \'pilot_input.json\').read_text())\n    manifest = (root / \'split_manifest.csv\').read_text()\n    if hashlib.md5(manifest.encode()).hexdigest() != FINGERPRINT:\n        raise ValueError(\'Unreviewed data split\')\n    if (root / \'test\').exists():\n        raise ValueError(\'Study input must exclude test images\')\n    rows = list(csv.DictReader(manifest.splitlines()))\n    scene_splits = defaultdict(set)\n    for row in rows:\n        scene_splits[row[\'scene\']].add(row[\'split\'])\n    if any(len(splits) > 1 for splits in scene_splits.values()):\n        raise ValueError(\'Scene leakage\')\n    weights = root / \'yolov8n_v4.pt\'\n    if hashlib.sha256(weights.read_bytes()).hexdigest() != metadata[\'weights_sha256\']:\n        raise ValueError(\'Unexpected baseline checkpoint\')\n    train = sorted((root / \'train/images\').glob(\'*\'))\n    val = sorted((root / \'valid/images\').glob(\'*\'))\n    if len(train) != 8247 or len(val) != 925:\n        raise ValueError(\'Unexpected input membership\')\n    scenes = {r[\'stem\']: r[\'scene\'] for r in rows if r[\'split\'] == \'train\'}\n    if set(scenes) != {p.stem for p in train}:\n        raise ValueError(\'Train manifest mismatch\')\n    eligible = train if args.arm == \'control\' else [p for p in train if scenes[p.stem] not in QUARANTINE]\n    rng = random.Random(args.seed)\n    if args.arm == \'curated-balanced\':\n        groups, negatives = defaultdict(list), []\n        for p in eligible:\n            label = root / \'train/labels\' / (p.stem + \'.txt\')\n            if any(line.split()[0] == \'2\' for line in label.read_text().splitlines() if line.strip()):\n                groups[scenes[p.stem]].append(p)\n            else:\n                negatives.append(p)\n        schedule = balanced_draws(groups, negatives, len(train), args.seed)\n    else:\n        schedule = eligible + rng.choices(eligible, k=len(train) - len(eligible))\n        rng.shuffle(schedule)\n    if len(schedule) != len(train):\n        raise ValueError(\'Unequal training budget\')\n    out = args.output / f\'study_{args.arm}_seed{args.seed}_{time.strftime("%Y%m%d_%H%M%S")}\'\n    out.mkdir(parents=True, exist_ok=False)\n    listing = out / \'training_draws.txt\'\n    listing.write_text(\'\'.join(str(p) + \'\\n\' for p in schedule))\n    config = out / \'data.yaml\'\n    config.write_text(yaml.safe_dump(dict(path=str(root), train=str(listing),\n                                          val=\'valid/images\', nc=5, names=NAMES)))\n    plan = dict(arm=args.arm, seed=args.seed, epochs=args.epochs, training_draws=len(schedule),\n        unique_training_images=len(set(schedule)), original_training_images=len(train),\n        quarantined_scenes=[] if args.arm == \'control\' else sorted(QUARANTINE),\n        updates_per_epoch=(len(schedule) + 15)//16, batch=16, optimizer=\'AdamW\', lr0=.0001,\n        patience=0, close_mosaic=0, test_used=False, validation_changed=False,\n        mosaic=.25, scale=.2, translate=.05, warmup_epochs=1., warmup_bias_lr=.0001,\n        cos_lr=True, lrf=.1, checkpoint_selection=\'fixed gate then Pillar recall, precision, overall AP\',\n        labels_changed=False, fraction_pillar=.25 if args.arm == \'curated-balanced\' else None,\n        max_repeats_pillar=4 if args.arm == \'curated-balanced\' else None,\n        checkpoint_sha256=metadata[\'weights_sha256\'], split_manifest_md5=FINGERPRINT,\n        training_draws_sha256=hashlib.sha256(listing.read_bytes()).hexdigest())\n    (out / \'plan.json\').write_text(json.dumps(plan, indent=2))\n    baseline = YOLO(str(weights))\n    if list(baseline.names.values()) != NAMES:\n        raise ValueError(\'Wrong class IDs\')\n    before = evaluate(baseline, root, out, \'before_validation\')\n    del baseline\n    torch.cuda.empty_cache()\n    model = YOLO(str(weights))\n    epoch_times = []\n    def epoch_start(trainer):\n        trainer._study_started = time.monotonic()\n    def epoch_end(trainer):\n        epoch_times.append(time.monotonic()-trainer._study_started)\n        print(\'STUDY_EPOCH\',json.dumps(dict(arm=args.arm,epoch=trainer.epoch+1,\n            seconds=epoch_times[-1],remaining_training_minutes=(args.epochs-trainer.epoch-1)*\n            sum(epoch_times)/len(epoch_times)/60)),flush=True)\n    def verify_budget(trainer):\n        if len(trainer.train_loader.dataset) != len(schedule):\n            raise ValueError(\'Loader dropped training entries; matched budget no longer holds\')\n    model.add_callback(\'on_train_start\', verify_budget)\n    model.add_callback(\'on_train_epoch_start\',epoch_start)\n    model.add_callback(\'on_train_epoch_end\',epoch_end)\n    model.train(data=str(config), epochs=args.epochs, patience=0, batch=16, imgsz=640,\n        device=0, seed=args.seed, deterministic=True, workers=2, optimizer=\'AdamW\',\n        lr0=.0001, close_mosaic=0, mosaic=.25, scale=.2, translate=.05,\n        warmup_epochs=1., warmup_bias_lr=.0001, cos_lr=True, lrf=.1,\n        project=str(out), name=\'candidate\', save_period=1)\n    del model\n    torch.cuda.empty_cache()\n    candidates = []\n    # Evaluate saved epochs at the real deployment threshold. Overall AP\'s\n    # best.pt alone can hide an otherwise useful weak-class checkpoint.\n    checkpoints = sorted((out/\'candidate/weights\').glob(\'epoch*.pt\'))\n    if not checkpoints:\n        raise ValueError(\'Per-epoch checkpoints were not saved\')\n    for checkpoint in checkpoints:\n        candidate = YOLO(str(checkpoint))\n        metrics = evaluate(candidate,root,out,\'validation_\'+checkpoint.stem)\n        passed, regressions = acceptance(before,metrics)\n        candidates.append(dict(checkpoint=str(checkpoint.relative_to(out)),metrics=metrics,\n                               gate_passed=passed,other_class_regressions=regressions))\n        print(\'STUDY_CHECKPOINT\',json.dumps(candidates[-1]),flush=True)\n        del candidate\n        torch.cuda.empty_cache()\n    selected = max(candidates,key=lambda c:(c[\'gate_passed\'],\n        c[\'metrics\'][\'per_class\'][\'Pillar\'][\'recall\'],\n        c[\'metrics\'][\'per_class\'][\'Pillar\'][\'precision\'],c[\'metrics\'][\'map50\']))\n    after,passed,regressions = selected[\'metrics\'],selected[\'gate_passed\'],selected[\'other_class_regressions\']\n    result = dict(plan=plan, before=before, after=after, gate_passed=passed,\n        other_class_regressions=regressions, application_changed=False,\n        selected_checkpoint=selected[\'checkpoint\'],checkpoints=candidates,epoch_seconds=epoch_times,\n        limitation=\'One seed is exploratory. Compare all arms at matched seeds, then confirm on untouched field data.\')\n    (out / \'study_result.json\').write_text(json.dumps(result, indent=2))\n    print(json.dumps(result, indent=2), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')

work = Path('/kaggle/working')
state = {'seed':0,'arms':['control','curated','curated-balanced'],'test_used':False,
         'application_changed':False,'status':'running','completed':[]}
def save_state():
    (work/'controlled_run_status.json').write_text(json.dumps(state,indent=2))
save_state()
try:
    for arm in state['arms']:
        print('START_ARM',arm,flush=True)
        subprocess.run([sys.executable,'-u','controlled_pillar_study.py','--arm',arm,
                        '--seed','0','--epochs','10'],check=True)
        state['completed'].append(arm)
        save_state()
    state['status']='complete'
except Exception as exc:
    state['status']='failed'
    state['error']=str(exc)
    raise
finally:
    save_state()
    review=work/'review_bundle'
    review.mkdir(exist_ok=True)
    shutil.copy2(work/'controlled_run_status.json',review/'controlled_run_status.json')
    results=[]
    for path in sorted(work.glob('study_*/study_result.json')):
        result=json.loads(path.read_text())
        arm=result['plan']['arm']
        folder=review/arm
        folder.mkdir(exist_ok=True)
        shutil.copy2(path,folder/'study_result.json')
        checkpoint=path.parent/result['selected_checkpoint']
        shutil.copy2(checkpoint,folder/'selected.pt')
        result['selected_sha256']=hashlib.sha256(checkpoint.read_bytes()).hexdigest()
        for name in ('plan.json','training_draws.txt','candidate/results.csv','candidate/args.yaml'):
            src=path.parent/name
            if src.exists(): shutil.copy2(src,folder/src.name)
        results.append(result)
    (review/'comparison.json').write_text(json.dumps(results,indent=2))
    shutil.make_archive(str(work/'visionaid_controlled_review'),'zip',review)
    print('FINAL_STUDY_STATUS',json.dumps(state),flush=True)
